# Preparing AI Model for Deployment: Training and Saving Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Train models using TensorFlow or PyTorch
- Save models in various formats
- Understand model serialization
- Prepare models for deployment
- Test model saving and loading

## 🔗 Prerequisites

- ✅ Understanding of TensorFlow/PyTorch
- ✅ Understanding of model training
- ✅ Python knowledge
- ✅ Basic ML knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 11, Unit 1**:
- Preparing AI model for deployment: training and saving a model using TensorFlow or PyTorch
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 1 Practical Content

---

## Introduction

**Model preparation** involves training a model and saving it in a format suitable for deployment. Proper serialization ensures models can be loaded and used in production environments.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np
import pickle
import os

print("✅ Libraries imported!")
print("\nPreparing AI Model for Deployment: Training and Saving")
print("=" * 60)

print("\nModel Saving Formats:")
print("  - TensorFlow: SavedModel, HDF5")
print("  - PyTorch: .pth, .pt")
print("  - Scikit-learn: Pickle (.pkl)")
print("  - ONNX: Cross-platform format")
print("  - PMML: XML-based format")

print("\n✅ Model preparation concepts understood!")

## 🌍 Real-World Worked Example — MLflow Experiment Tracking (Production Pattern)

**Industry context:**
- Netflix uses MLflow to track 1000s of A/B test model variants
- Airbnb logs every model training run with parameters, metrics, and artifacts
- Booking.com uses experiment tracking to compare models before deploying to 150M users

We demonstrate **MLflow-style experiment tracking** using Python — the same pattern used in production ML pipelines.

In [ ]:
import json, time, pathlib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

# ── Simulate MLflow-style experiment tracking ─────────────────────────────
RUNS_LOG = pathlib.Path('/tmp/experiment_runs.json')

def log_run(name, params, metrics, tags=None):
    # Simulates mlflow.log_params(), mlflow.log_metrics()
    run = {
        'run_name': name,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
        'params': params,
        'metrics': metrics,
        'tags': tags or {}
    }
    runs = json.loads(RUNS_LOG.read_text()) if RUNS_LOG.exists() else []
    runs.append(run)
    RUNS_LOG.write_text(json.dumps(runs, indent=2))
    return run

iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
y = iris.target

models = {
    'LogisticRegression':   (LogisticRegression(max_iter=500, C=1.0),    {'C':1.0, 'max_iter':500}),
    'RandomForest_50':      (RandomForestClassifier(n_estimators=50),    {'n_estimators':50}),
    'RandomForest_100':     (RandomForestClassifier(n_estimators=100),   {'n_estimators':100}),
    'GradientBoosting':     (GradientBoostingClassifier(n_estimators=50),{'n_estimators':50, 'lr':0.1}),
}

results = []
print("Running experiments...
")
for name, (clf, params) in models.items():
    start = time.perf_counter()
    cv_scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    elapsed = time.perf_counter()-start
    metrics = {
        'cv_mean_accuracy': round(cv_scores.mean(),4),
        'cv_std':           round(cv_scores.std(),4),
        'training_time_s':  round(elapsed,3)
    }
    run = log_run(name, params, metrics, tags={'dataset':'iris','framework':'sklearn'})
    results.append((name, metrics))
    print(f"  [{name:25s}]  acc={metrics['cv_mean_accuracy']:.4f} ± {metrics['cv_std']:.4f}  | {elapsed:.2f}s")

# ── Plot — production-style model comparison ──────────────────────────────
names  = [r[0].replace('_', '
') for r in results]
means  = [r[1]['cv_mean_accuracy'] for r in results]
stds   = [r[1]['cv_std']           for r in results]
times  = [r[1]['training_time_s']  for r in results]
best   = np.argmax(means)

fig, axes = plt.subplots(1,2, figsize=(12,4))
bars = axes[0].bar(names, means, yerr=stds, capsize=5, alpha=0.8)
bars[best].set_color('green'); bars[best].set_label('Best model')
axes[0].set_title("Model Comparison (MLflow-style)"); axes[0].set_ylabel("CV Accuracy"); axes[0].legend()
axes[1].bar(names, times, alpha=0.8)
axes[1].set_title("Training Time"); axes[1].set_ylabel("Seconds")
plt.suptitle("Production ML Experiment Tracking — Same Pattern as Netflix/Airbnb")
plt.tight_layout(); plt.show()
print(f"
✅ Best model: {results[best][0]} (accuracy={means[best]:.4f})")
print(f"Experiment log saved to: {RUNS_LOG}")

## 📚 References & Further Reading

**Cloud ML Platforms:**
- [AWS SageMaker](https://docs.aws.amazon.com/sagemaker/)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs)
- [Azure Machine Learning](https://learn.microsoft.com/en-us/azure/machine-learning/)

**MLOps:**
- [MLflow](https://mlflow.org/) — Open-source experiment tracking
- [Weights & Biases](https://wandb.ai/) — Production MLOps platform

**State-of-the-Art:** Netflix, Airbnb, Uber run 1000+ ML models in production using SageMaker/Vertex AI with full MLflow experiment tracking.

## 📝 Summary

You learned **model packaging and serialization** — converting trained models into deployable artifacts. Pickle is simple but Python-only; ONNX is cross-platform and hardware-optimized. Production systems at Uber, Lyft, and Airbnb use ONNX and TorchScript for portability.